In [ ]:
from pathlib import Path

import torch
from IPython.display import display
from lightning.pytorch import Trainer, seed_everything
from torchvision.transforms.functional import to_pil_image
from torchvision.utils import make_grid

from radio_diffusion.data.mnist import MNISTDataModule
from radio_diffusion.diffusion.ddpm import GaussianDiffusion
from radio_diffusion.models.unet import CompactUNet
from radio_diffusion.training.module import DDPMModule

## Data configuration

The kernel's working directory is assumed to be `notebooks/`, so `../data` refers to the repository's data directory. Setting `download=True` allows missing MNIST files to be downloaded.

In [ ]:
data_dir = Path("../data")
data = MNISTDataModule(
    data_dir=data_dir,
    batch_size=32,
    num_workers=0,
    val_fraction=0.1,
    seed=42,
    download=True,
)

## Data preparation

`prepare_data()` loads existing MNIST files or downloads missing data.

In [ ]:
data.prepare_data()

## Image batch

`setup("fit")` creates the training and validation split. Each batch contains grayscale images with shape `[batch, 1, 28, 28]`, stored as `torch.float32` tensors with pixel values scaled to `[-1, 1]`.

In [ ]:
data.setup("fit")
train_loader = data.train_dataloader()
images = next(iter(train_loader))

print(f"Training images: {len(data.train_dataset)}")
print(f"Validation images: {len(data.val_dataset)}")
print(f"Batch shape: {tuple(images.shape)}; dtype: {images.dtype}")
print(f"Pixel range: [{images.min().item():.3f}, {images.max().item():.3f}]")

## Sample images

The first 16 images are rescaled to `[0, 1]` and arranged in a grid for display.

In [ ]:
grid = make_grid((images[:16] + 1) / 2, nrow=4)
preview = to_pil_image(grid)
display(preview.resize((preview.width * 3, preview.height * 3)))

## Forward diffusion

The schedule assigns a noise variance $\beta_t$ to each step. The cumulative factor is $\bar{\alpha}_t = \prod_{s=0}^{t}(1 - \beta_s)$. A noisy image is computed directly from the original image:

$$x_t = \sqrt{\bar{\alpha}_t}\,x_{\mathrm{clean}} + \sqrt{1 - \bar{\alpha}_t}\,\epsilon, \qquad \epsilon \sim \mathcal{N}(0, I).$$

`GaussianDiffusion` stores the schedule and precomputed coefficients. Its `add_noise()` method returns both the noisy images and the sampled noise. The noise serves as the prediction target during training. This follows equation (4) of the [DDPM paper](https://arxiv.org/abs/2006.11239), with zero-based schedule indices: index 0 is already slightly noisy.

In [ ]:
diffusion = GaussianDiffusion(
    num_steps=1000,
    beta_start=1e-4,
    beta_end=0.02,
)

## Images at increasing noise levels

Each row shows the same digit, beginning with the original image. The same Gaussian noise is reused across the selected timesteps to make the change in noise strength visible. Each column is calculated directly from the clean images.

Noisy values can extend beyond `[-1, 1]`. Clipping is applied only to the display copy; the forward diffusion output retains its full range.

In [ ]:
torch.manual_seed(42)
clean_images = images[:4]
noise = torch.randn_like(clean_images)
timesteps_to_show = torch.linspace(
    0, diffusion.num_steps - 1, 5, dtype=torch.long
).tolist()
columns = [clean_images]

for timestep in timesteps_to_show:
    timesteps = torch.full(
        (len(clean_images),), timestep, dtype=torch.long, device=clean_images.device
    )
    noisy_images, _ = diffusion.add_noise(clean_images, timesteps, noise=noise)
    columns.append(noisy_images)

comparison = torch.stack(columns, dim=1).flatten(0, 1)
display_images = ((comparison + 1) / 2).clamp(0, 1)
grid = make_grid(display_images, nrow=len(columns))
preview = to_pil_image(grid.cpu())
print("Columns:", ["original", *timesteps_to_show])
display(preview.resize((preview.width * 3, preview.height * 3)))

## Model training

The U-Net is trained to recover the Gaussian noise used to construct each noisy image. `DDPMModule` samples a diffusion timestep for every image, applies forward diffusion, computes the mean squared error between the predicted and sampled noise, and updates the network. Validation uses the same objective without parameter updates.

In [ ]:
seed_everything(42, workers=True)

network = CompactUNet(
    in_channels=1,
    channels=(32, 64, 128),
    time_embedding_dim=64,
)
training_module = DDPMModule(
    model=network,
    diffusion=diffusion,
    learning_rate=1e-3,
)

trainer = Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    default_root_dir=Path("../outputs/mnist"),
    log_every_n_steps=50,
)

In [ ]:
trainer.fit(training_module, datamodule=data)

## Generate samples

Sampling starts from Gaussian noise and applies the learned reverse process from the final diffusion timestep to timestep zero. With the configured schedule, this requires 1,000 U-Net evaluations for the generated batch. The generated values are clipped only for display.

In [ ]:
generated_images = training_module.diffusion.sample(
    model=training_module.model,
    shape=(16, 1, 28, 28),
)

display_images = ((generated_images.cpu() + 1) / 2).clamp(0, 1)
grid = make_grid(display_images, nrow=4)
preview = to_pil_image(grid)
display(preview.resize((preview.width * 3, preview.height * 3)))